# OASIS3 Metadata Subject Cleaning

In [1]:
# Import necessary libraries:
import pandas as pd
import re
import os

# Read kept sessions file:
with open('logs/1.valid_subjects_sessions/kept_sessions_log_20260521_090340.txt', 'r') as f:
    lines = f.readlines()

# Extract subject + session day code (e.g. OAS30002_d0653):
kept_labels = []
seen = set()

# Iterate over the several lines in the text file:
for line in lines:
    # Extract the current subject and session name:
    line = line.strip()
    
    # Match patterns like: sub-OAS30002/ses-d0653
    match = re.search(r"sub-(OAS\d+)/ses-(d\d+)", line)
    
    # Proceed if a match was found:
    if match:
        # Extract the subject name:
        subj = match.group(1)

        # Extract the session name:
        ses = match.group(2)

        # Define the label text:
        label = f"{subj}_{ses}"

        if label not in seen:
            kept_labels.append(label)
            seen.add(label)

# Create a new clean metadata dataframe:
df_clean = pd.DataFrame({'Subject_ID': [x.split('_')[0] for x in kept_labels], 'Session_ID': [x.split('_')[1] for x in kept_labels]})

# Load OASIS3 metadata as a dataframe:
df_meta = pd.read_csv('../OASIS3_metadata.csv')

# Define the list of MCI columns:
mci_cols = df_meta.filter(regex='^MCI').columns.tolist()

# Define a list of desired columns:
cols = ['OASISID', 'days_to_visit', 'age at visit', 'NORMCOG', 'DEMENTED'] + mci_cols

# Create a metadata dataframe with only those columns:
df_sub = df_meta[cols]

# Clean the dataframe:
df_meta_clean = (
    df_sub
    .groupby('OASISID', as_index=False)
    .agg({
        'days_to_visit': 'first',
        'age at visit': 'first',

        # DEMENTED: if any 1 is found → 1
        'DEMENTED': lambda x: int((x == 1).any()),

        # MCI: if any 1 is found → 1 
        **{col: lambda x: int((x == 1).any()) for col in mci_cols},

        # NORMCOG: if any 1 is found → 1
        'NORMCOG': lambda x: int((x == 1).any())
    })
)

# If a subject is demented, force all MCI columns and NORMCOG to 0 (dementia overrides everything):
df_meta_clean.loc[df_meta_clean['DEMENTED'] == 1, mci_cols + ['NORMCOG']] = 0

# If a subject is not demented but any MCI column is 1, then NORMCOG and DEMENTED must be 0:
mask = (df_meta_clean['DEMENTED'] == 0) & (df_meta_clean[mci_cols].sum(axis=1) > 0)
df_meta_clean.loc[mask, mci_cols] = 1
df_meta_clean.loc[mask, 'NORMCOG'] = 0
df_meta_clean.loc[mask, 'DEMENTED'] = 0

# Calculate the average and create the new 'MCI' column:
df_meta_clean['MCI'] = df_meta_clean[mci_cols].mean(axis=1, skipna=True)

# Replace NaN values with zeros:
df_meta_clean['MCI'] = df_meta_clean['MCI'].fillna(0)

# Drop the original individual MCI columns from the DataFrame:
df_meta_clean.drop(columns=mci_cols, inplace=True)

# Create a new days to visit column in the clean metadata dataframe:
df_clean['days_to_visit'] = df_clean['Session_ID'].str.extract(r'd(\d+)').astype(int)

# Merge the clean dataframe with the clean metadata so each subject gets their reference point (a known (ref_days, ref_age) pair to interpolate from):
df_clean = df_clean.merge(df_meta_clean.rename(columns={'OASISID':'Subject_ID','days_to_visit':'ref_days','age at visit':'ref_age'}), on='Subject_ID')

# Calculate the age of the corresponding session for each subject:
df_clean['Age'] = df_clean['ref_age'] + (df_clean['days_to_visit'] - df_clean['ref_days']) / 365.25

# Round up the age column:
df_clean['Age'] = df_clean['Age'].round(2)

# Remove the reference columns:
df_clean = df_clean.drop(columns=['ref_days','ref_age','days_to_visit'])

In [2]:
# Make a list of the subjects kept:
subjects_kept = sorted(set(df_clean['Subject_ID']))
subjects_kept = [f"sub-{s}" for s in subjects_kept]

# Make a list of all discarded subjects:
all_subjects = sorted(set([x.split('_')[0] for x in kept_labels]))
subjects_discarded = sorted(set(all_subjects) - set(df_clean['Subject_ID']))
subjects_discarded = [f"sub-{s}" for s in subjects_discarded]

# Create output folder:
out_dir = "logs/2.valid_subjects_metadata"
os.makedirs(out_dir, exist_ok=True)

# Save kept subjects log:
kept_path = f"{out_dir}/kept_subjects_in_metadata_log.txt"
with open(kept_path, "w") as f:
    f.write("\n".join(subjects_kept))

# Save deleted subjects log:
deleted_path = f"{out_dir}/deleted_subjects_not_in_metadata_log.txt"
with open(deleted_path, "w") as f:
    f.write("\n".join(subjects_discarded))

# Prepare the summary text:
summary_text = (
    "| SUMMARY |\n"
    f"Total subjects processed: {len(all_subjects)}\n"
    f"Subjects present in metadata kept: {len(subjects_kept)}\n"
    f"Subjects not present in metadata deleted: {len(subjects_discarded)}\n"
)

# Print to console:
print(summary_text)

# Save to file:
summary_file = f"{out_dir}/summary_metadata_subjects.txt"
with open(summary_file, "w") as f:
    f.write(summary_text)

| SUMMARY |
Total subjects processed: 1188
Subjects present in metadata kept: 1154
Subjects not present in metadata deleted: 34



In [3]:
# Define all MCI-related columns:
mci_cols = [col for col in df_clean.columns if col.startswith('MCI')]

# Row-wise check: does this subject have ANY label = 1?
df_clean['HAS_LABEL'] = ((df_clean['NORMCOG'] == 1) | (df_clean['DEMENTED'] == 1)| (df_clean[mci_cols].eq(1).any(axis=1)))

# Now each subject contributes at most 1:
total_subjects_with_label = df_clean['HAS_LABEL'].sum()

# Subjects with label:
label_df = df_clean[df_clean['HAS_LABEL']]

# List of subject IDs with no label:
label_subjects = label_df['Subject_ID'].tolist()
label_subjects = [f"sub-{s}" for s in label_subjects]

# Subjects with no label at all:
no_label_df = df_clean[~df_clean['HAS_LABEL']]

# List of subject IDs with no label:
no_label_subjects = no_label_df['Subject_ID'].tolist()
no_label_subjects = [f"sub-{s}" for s in no_label_subjects]

# Remove has label column:
label_df = label_df.drop(columns=['HAS_LABEL'])

# Create output folder:
out_dir = "logs/3.valid_subjects_label"
os.makedirs(out_dir, exist_ok=True)

# Save kept subjects log:
kept_path = f"{out_dir}/kept_subjects_with_label_log.txt"
with open(kept_path, "w") as f:
    f.write("\n".join(label_subjects))

# Save deleted subjects log:
deleted_path = f"{out_dir}/deleted_subjects_without_label_log.txt"
with open(deleted_path, "w") as f:
    f.write("\n".join(no_label_subjects))

# Prepare the summary text:
summary_text = (
    "| SUMMARY |\n"
    f"Total subjects processed: {len(df_clean)}\n"
    f"Subjects with label kept: {len(label_subjects)}\n"
    f"Subjects with no labels deleted: {len(no_label_subjects)}\n"
)

# Print to console:
print(summary_text)

# Save to file:
summary_file = f"{out_dir}/summary_label_subjects.txt"
with open(summary_file, "w") as f:
    f.write(summary_text)

| SUMMARY |
Total subjects processed: 1154
Subjects with label kept: 1149
Subjects with no labels deleted: 5



In [4]:
# Create a boolean mask for rows where any MCI column >= 1:
mci_mask = (label_df[mci_cols] >= 1).any(axis=1)

# Count of MCI subjects removed:
num_mci_subjects = mci_mask.sum()

# List of subject IDs to keep:
subjects_kept = label_df.loc[~mci_mask, 'Subject_ID'].tolist()
subjects_kept = [f"sub-{s}" for s in subjects_kept]

# List of subject IDs removed:
mci_removed_subjects = label_df.loc[mci_mask, 'Subject_ID'].tolist()
mci_removed_subjects = [f"sub-{s}" for s in mci_removed_subjects]

# Remove MCI column:
label_df = label_df.drop(columns=['MCI'])

# Create output folder:
out_dir = "logs/4.valid_subjects_no_mci"
os.makedirs(out_dir, exist_ok=True)

# Save kept subjects log:
kept_path = f"{out_dir}/kept_subjects_no_mci_log.txt"
with open(kept_path, "w") as f:
    f.write("\n".join(subjects_kept))

# Save kept subjects log:
final_kept_path = f"../final_subjects.txt"
with open(final_kept_path, "w") as f:
    f.write("\n".join(subjects_kept))

# Save deleted subjects log:
deleted_path = f"{out_dir}/deleted_subjects_mci_log.txt"
with open(deleted_path, "w") as f:
    f.write("\n".join(mci_removed_subjects))

# Prepare the summary text:
summary_text = (
    "| SUMMARY |\n"
    f"Total subjects processed: {len(label_df)}\n"
    f"Subjects with no MCI kept: {len(subjects_kept)}\n"
    f"Subjects with MCI deleted: {len(mci_removed_subjects)}\n"
)

# Print to console:
print(summary_text)

# Save to file:
summary_file = f"{out_dir}/summary_no_mci_subjects.txt"
with open(summary_file, "w") as f:
    f.write(summary_text)

| SUMMARY |
Total subjects processed: 1149
Subjects with no MCI kept: 1094
Subjects with MCI deleted: 55



In [5]:
# Define final dataframe to keep:
df_final = label_df.loc[~mci_mask]

# Save final CSV:
df_final.to_csv('../OASIS3_metadata_clean.csv', index=False)